# Celebal Technologies Summer Internship 2026
## Silver Layer: Data Cleaning & Transformation
### FMCG Data Consolidation & Analytics Platform


**Objective:** Clean, standardize, and merge data from both companies into a unified Silver layer.

**Operations:**
- Handle null values
- Remove duplicates
- Standardize schemas
- Merge both companies data
- Normalize categories

In [0]:

# SILVER LAYER: Data Cleaning & Transformation
# Medallion Architecture - Layer 2

from pyspark.sql.functions import (
    col, upper, trim, when, expr,
    sum as spark_sum, to_date, lit
)

print("=" * 60)
print("SILVER LAYER: Transformation Started")
print("=" * 60)

# Create Silver database
spark.sql("CREATE DATABASE IF NOT EXISTS silver_fmcg")

# Load Bronze data
df_bronze_a = spark.read.table("bronze_fmcg.company_a_raw")
df_bronze_b = spark.read.table("bronze_fmcg.company_b_raw")

print(f"Company A loaded: {df_bronze_a.count()} rows")
print(f"Company B loaded: {df_bronze_b.count()} rows")

SILVER LAYER: Transformation Started
Company A loaded: 9994 rows
Company B loaded: 15 rows


In [0]:

# Clean Company A

def clean_company_a(df):
    """
    Cleans Company A (Superstore) data:
    - Safe cast numeric columns
    - Remove duplicates
    - Handle nulls
    - Standardize text
    """
    print("=== Cleaning Company A ===")

    # Safe cast numeric columns
    df = df \
        .withColumn("Sales", expr("try_cast(Sales as double)")) \
        .withColumn("Quantity", expr("try_cast(Quantity as double)")) \
        .withColumn("Discount", expr("try_cast(Discount as double)")) \
        .withColumn("Profit", expr("try_cast(Profit as double)"))

    # Check nulls
    print("Null counts:")
    df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in ["customer_id", "Sales", "Quantity", "Discount", "Profit"]
    ]).show()

    # Remove duplicates
    before = df.count()
    df = df.dropDuplicates(["order_id", "product_id"])
    after = df.count()
    print(f"Duplicates removed: {before - after}")

    # Fill nulls
    df = df.na.fill(0.0, ["Sales", "Quantity", "Discount", "Profit"])
    df = df.na.fill("Unknown", ["customer_id", "customer_name", "Segment"])

    # Standardize text
    df = df \
        .withColumn("Segment", trim(upper(col("Segment")))) \
        .withColumn("Region", trim(upper(col("Region")))) \
        .withColumn("Category", trim(col("Category")))

    print(f"Company A cleaned: {df.count()} rows")
    return df

df_silver_a = clean_company_a(df_bronze_a)
df_silver_a.show(5)

=== Cleaning Company A ===
Null counts:
+-----------+-----+--------+--------+------+
|customer_id|Sales|Quantity|Discount|Profit|
+-----------+-----+--------+--------+------+
|          0|  300|      20|      11|     0|
+-----------+-----+--------+--------+------+

Duplicates removed: 8
Company A cleaned: 9986 rows
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+--------------+--------------+--------------------+---------+
|row_id|      order_id|order_date| ship_date|     ship_mode|customer_id|  customer_name|  Segment|      Country|           City|     State|postal_code|Region|     product_id|       Category|sub_category|        product_name|   Sales|Quantity|Discount|  Profit|source_company| source_system| ingestion_timestamp| batch_id|
+------+--------------+----------+-----

In [0]:

# Clean Company B

def clean_company_b(df):
    """
    Cleans Company B (Legacy FMCG) data:
    - Handle nulls
    - Rename columns to match Company A schema
    - Map categories to unified taxonomy
    - Standardize text
    """
    print("=== Cleaning Company B ===")

    # Check nulls
    print("Null counts:")
    df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in ["cust_code", "order_ref", "revenue", "units_sold"]
    ]).show()

    # Fill nulls
    df = df.na.fill("UNKNOWN", ["cust_code", "cust_name"])
    df = df.na.fill(0.0, ["revenue", "discount_rate", "net_profit"])

    # Rename columns to match Company A schema
    df = df \
        .withColumnRenamed("order_ref", "order_id") \
        .withColumnRenamed("cust_code", "customer_id") \
        .withColumnRenamed("cust_name", "customer_name") \
        .withColumnRenamed("business_type", "Segment") \
        .withColumnRenamed("zone", "Region") \
        .withColumnRenamed("product_category", "Category") \
        .withColumnRenamed("revenue", "Sales") \
        .withColumnRenamed("units_sold", "Quantity") \
        .withColumnRenamed("discount_rate", "Discount") \
        .withColumnRenamed("net_profit", "Profit")

    # Map categories to unified taxonomy
    df = df.withColumn("Category",
        when(col("Category") == "Soap & Detergent", "Home Care")
        .when(col("Category") == "Personal Care", "Personal Care")
        .when(col("Category") == "Beverages", "Food & Beverages")
        .when(col("Category") == "Snacks", "Food & Beverages")
        .otherwise(col("Category"))
    )

    # Standardize text
    df = df \
        .withColumn("Segment", trim(upper(col("Segment")))) \
        .withColumn("Region", trim(upper(col("Region")))) \
        .withColumn("Category", trim(col("Category"))) \
        .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
        .withColumn("ship_date", col("order_date")) \
        .withColumn("Country", lit("India")) \
        .withColumn("sub_category", col("Category"))

    print(f"Company B cleaned: {df.count()} rows")
    df.show(5)
    return df

df_silver_b = clean_company_b(df_bronze_b)

=== Cleaning Company B ===
Null counts:
+---------+---------+-------+----------+
|cust_code|order_ref|revenue|units_sold|
+---------+---------+-------+----------+
|        1|        0|      0|         0|
+---------+---------+-------+----------+

Company B cleaned: 15 rows
+----------+----------+-----------+---------------+---------+-----------+-------+----------------+------------------+------+--------+--------+------+--------------+------------------+--------------------+---------+----------+-------+----------------+
|  order_id|order_date|customer_id|  customer_name|  Segment|     Region|   city|        Category|      product_name| Sales|Quantity|Discount|Profit|source_company|     source_system| ingestion_timestamp| batch_id| ship_date|Country|    sub_category|
+----------+----------+-----------+---------------+---------+-----------+-------+----------------+------------------+------+--------+--------+------+--------------+------------------+--------------------+---------+----------+

In [0]:

# Merge Both Companies


# Select common columns
df_silver_a_select = df_silver_a.select(
    col("order_id"), col("order_date"), col("ship_date"),
    col("customer_id"), col("customer_name"), col("Segment"),
    col("Region"), col("City").alias("city"), col("Country"),
    col("Category"), col("sub_category"), col("product_name"),
    col("Sales"), col("Quantity"), col("Discount"), col("Profit"),
    col("source_company")
)

df_silver_b_select = df_silver_b.select(
    col("order_id"), col("order_date"), col("ship_date"),
    col("customer_id"), col("customer_name"), col("Segment"),
    col("Region"), col("city"), col("Country"),
    col("Category"), col("sub_category"), col("product_name"),
    col("Sales"), col("Quantity"), col("Discount"), col("Profit"),
    col("source_company")
)

# Union both
df_silver_merged = df_silver_a_select.union(df_silver_b_select)

print("=== SILVER LAYER: Merged Dataset ===")
print(f"Company A rows: {df_silver_a_select.count()}")
print(f"Company B rows: {df_silver_b_select.count()}")
print(f"Total merged: {df_silver_merged.count()}")
df_silver_merged.show(5)

# Drop and recreate
spark.sql("DROP TABLE IF EXISTS silver_fmcg.unified_sales")
df_silver_merged.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_fmcg.unified_sales")

print("\n silver_fmcg.unified_sales saved!")
print(f"Total rows: {spark.read.table('silver_fmcg.unified_sales').count()}")

=== SILVER LAYER: Merged Dataset ===
Company A rows: 9986
Company B rows: 15
Total merged: 10001
+--------------+----------+----------+-----------+---------------+--------+-------+------------+-------------+---------------+------------+--------------------+-------+--------+--------+-------+--------------+
|      order_id|order_date| ship_date|customer_id|  customer_name| Segment| Region|        city|      Country|       Category|sub_category|        product_name|  Sales|Quantity|Discount| Profit|source_company|
+--------------+----------+----------+-----------+---------------+--------+-------+------------+-------------+---------------+------------+--------------------+-------+--------+--------+-------+--------------+
|CA-2014-115812|2014-06-09|2014-06-14|   BH-11710|Brosina Hoffman|CONSUMER|   WEST| Los Angeles|United States|     Technology|      Phones|Mitel 5320 IP Pho...|907.152|     6.0|     0.2|90.7152|     Company_A|
|US-2017-156909|2017-07-16|2017-07-18|   SF-20065|Sandra Flanag